# Quality Data Preprocessing (MPP=0.5 기준)

Visium HD, Visium HD 3', Xenium 3개 기술만 대상으로  
MPP=0.5 µm/pixel (20x) 기준 grid 패치 추출 + 2-head label 생성

In [ ]:
# ===== 1. 설정 및 라이브러리 =====
from glob import glob
import os
import numpy as np
import pandas as pd
import anndata as ad
import openslide
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore', message='Variable names are not unique')

def create_dir(path):
    os.makedirs(path, exist_ok=True)

# 하이퍼파라미터
TARGET_MPP = 0.05        # 20x 해상도 (µm/pixel)
CONTEXT_SCALE = 4       # 5x context = 4x wider
PATCH_SIZE = 512        # 패치 크기 (pixels)
MIN_SPOTS = 1           # 패치 내 최소 spot 수

# 대상 기술
TARGET_TECHS = ['Visium HD', "Visium HD 3'", 'Xenium']

# Marker genes
MARKER_GENES = [
    'EPCAM', 'KRT8', 'KRT18', 'KRT19',
    'COL1A1', 'COL1A2', 'FAP', 'ACTA2',
    'CD3D', 'CD3E', 'CD8A',
    'MS4A1', 'CD79A',
    'CD68',
    'PECAM1'
]
NUM_GENES = len(MARKER_GENES)

# Gene alias dictionary (Xenium 패널 등에서 이름이 다를 수 있음)
GENE_ALIAS = {
    "EPCAM": ["EPCAM", "TACSTD1", "GA733-2", "CD326", "EPCAM_HUMAN"],
    "KRT8": ["KRT8", "CK8", "CYK8", "K8"],
    "KRT18": ["KRT18", "CK18", "CYK18", "K18"],
    "KRT19": ["KRT19", "CK19", "CYK19", "K19"],
    "COL1A1": ["COL1A1", "COL1A1_HUMAN"],
    "COL1A2": ["COL1A2", "COL1A2_HUMAN"],
}

def resolve_gene_name(marker_gene, gene_names_in_data):
    """marker_gene의 primary name 또는 alias 중 데이터에 존재하는 이름을 반환.
    찾지 못하면 None 반환."""
    if marker_gene in gene_names_in_data:
        return marker_gene
    if marker_gene in GENE_ALIAS:
        for alias in GENE_ALIAS[marker_gene]:
            if alias in gene_names_in_data:
                return alias
    return None

print(f'TARGET_MPP: {TARGET_MPP} µm/pixel')
print(f'PATCH_SIZE: {PATCH_SIZE}')
print(f'CONTEXT_SCALE: {CONTEXT_SCALE} (5x)')
print(f'TARGET_TECHS: {TARGET_TECHS}')
print(f'GENE_ALIAS keys: {list(GENE_ALIAS.keys())}')

In [ ]:
# ===== 2. 데이터 경로 및 저장 디렉토리 생성 =====
data_path = '../../data/spatialTranscriptome/'
save_base = '../../data/marker_gene_spatial_expression'

meta_df = pd.read_csv(f'{data_path}meta_df_homo_sapiens.csv')
print(f'Total slides in metadata: {len(meta_df)}')

# 대상 기술만 필터링
target_df = meta_df[meta_df['st_technology'].isin(TARGET_TECHS)].reset_index(drop=True)
print(f'Target slides ({TARGET_TECHS}): {len(target_df)}')
print(target_df['st_technology'].value_counts())

# organ 목록
organs = target_df['organ'].unique().tolist()
print(f'Organs: {organs}')

# 저장 디렉토리 생성
for o in organs:
    for t in TARGET_TECHS:
        create_dir(f'{save_base}/image/{t}/{o}')
        create_dir(f'{save_base}/image_5x/{t}/{o}')
        create_dir(f'{save_base}/label_proportion/{t}/{o}')
        create_dir(f'{save_base}/label_intensity/{t}/{o}')

print('Directories created.')

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='Variable names are not unique')
from tqdm import tqdm

marker_genes = [
    'EPCAM', 'KRT8', 'KRT18', 'KRT19',
    'COL1A1', 'COL1A2', 'FAP', 'ACTA2',
    'CD3D', 'CD3E', 'CD8A',
    'MS4A1', 'CD79A',
    'CD68',
    'PECAM1'
]

# ===== Pass 1: 전체 슬라이드에서 gene별 global percentile 계산 =====
gene_values = {g: [] for g in marker_genes}
valid_slide_ids = []

for idx in tqdm(range(len(target_df)), desc='Pass 1: Collecting gene stats'):
    row = target_df.iloc[idx]
    sample_id = row['id']

    h5ad_path = f'{data_path}st/{sample_id}.h5ad'
    if not os.path.exists(h5ad_path):
        continue

    adata = ad.read_h5ad(h5ad_path)
    adata.var_names_make_unique()
    gene_names = adata.var_names.tolist()

    # alias를 통해 실제 데이터 내 gene 이름 resolve
    resolved = {g: resolve_gene_name(g, gene_names) for g in marker_genes}
    missing = [g for g, r in resolved.items() if r is None]
    if len(missing) > 0:
        continue

    valid_slide_ids.append(sample_id)

    # resolved된 이름으로 인덱스 추출
    gene_indices = [gene_names.index(resolved[g]) for g in marker_genes]
    if hasattr(adata.X, 'toarray'):
        expr_matrix = adata.X[:, gene_indices].toarray()
    else:
        expr_matrix = np.array(adata.X[:, gene_indices])

    # alias로 매칭된 gene이 있으면 알림
    aliased = {g: resolved[g] for g in marker_genes if resolved[g] != g}
    if aliased:
        print(f'  {sample_id}: alias matched: {aliased}')

    for gi, gene in enumerate(marker_genes):
        # 0이 아닌 발현값만 수집 (sparse 데이터 특성상 0이 대부분)
        vals = expr_matrix[:, gi]
        gene_values[gene].append(vals)

# gene별 p1, p99 계산
gene_p1 = {}
gene_p99 = {}
for gene in marker_genes:
    all_vals = np.concatenate(gene_values[gene])
    gene_p1[gene] = np.percentile(all_vals, 1)
    gene_p99[gene] = np.percentile(all_vals, 99)
    print(f'{gene:8s} | p1={gene_p1[gene]:.4f}, p99={gene_p99[gene]:.4f}')

print(f'\nValid slides: {len(valid_slide_ids)} / {len(target_df)}')

# global stats 저장 (재사용 가능)
global_stats = np.array([[gene_p1[g], gene_p99[g]] for g in marker_genes])
np.save(f'{data_path}global_gene_stats_p1_p99.npy', global_stats)
print(f'Saved: {data_path}global_gene_stats_p1_p99.npy')

In [ ]:
# ===== 4. Grid 패치 추출 + 2-head label (MPP=0.5 기준) =====
from concurrent.futures import ThreadPoolExecutor
import glob as glob_module

save_image_dir = f'{save_base}/image'
save_image_5x_dir = f'{save_base}/image_5x'
save_proportion_dir = f'{save_base}/label_proportion'
save_intensity_dir = f'{save_base}/label_intensity'

# 이미 처리된 슬라이드 스킵
existing_files = set()
for f in glob_module.glob(f'{save_image_dir}/**/*.tiff', recursive=True):
    basename = os.path.splitext(os.path.basename(f))[0]
    parts = basename.rsplit('_', 2)
    if len(parts) == 3:
        existing_files.add(parts[0])
print(f'Already processed slides: {len(existing_files)}')

skipped = []
processed = []

for idx in tqdm(range(len(target_df)), desc='Grid extraction (MPP=0.5)'):
    row = target_df.iloc[idx]
    sample_id = row['id']
    organ_name = row['organ']
    st_tech = row['st_technology']
    image_file = row['image_filename']
    patch_count = 0

    if sample_id in existing_files:
        continue

    # h5ad 로드
    h5ad_path = f'{data_path}st/{sample_id}.h5ad'
    if not os.path.exists(h5ad_path):
        skipped.append((sample_id, 'h5ad not found'))
        continue

    adata = ad.read_h5ad(h5ad_path)
    adata.var_names_make_unique()
    gene_names = adata.var_names.tolist()

    # alias를 통해 실제 데이터 내 gene 이름 resolve
    resolved = {g: resolve_gene_name(g, gene_names) for g in MARKER_GENES}
    missing = [g for g, r in resolved.items() if r is None]
    if len(missing) >= 7:
        skipped.append((sample_id, f'too many missing genes ({len(missing)}): {missing}'))
        continue

    # alias로 매칭된 gene 알림
    aliased = {g: resolved[g] for g in MARKER_GENES if resolved[g] is not None and resolved[g] != g}
    if aliased:
        print(f'  {sample_id}: alias matched: {aliased}')

    # missing gene이 있으면 알림
    if len(missing) > 0:
        print(f'  {sample_id}: {len(missing)} missing genes (filled with 0): {missing}')

    # 존재하는 gene의 인덱스만 추출 (resolved 이름 사용)
    available_genes = [g for g in MARKER_GENES if resolved[g] is not None]
    available_gene_indices = [gene_names.index(resolved[g]) for g in available_genes]
    available_marker_indices = [MARKER_GENES.index(g) for g in available_genes]

    # WSI 로드
    wsi_path = f'{data_path}wsis/{image_file}'
    if not os.path.exists(wsi_path):
        skipped.append((sample_id, 'WSI not found'))
        continue

    slide = openslide.open_slide(wsi_path)
    slide_w, slide_h = slide.dimensions

    # MPP를 이미지 헤더에서 가져오기
    slide_mpp = float(slide.properties.get('openslide.mpp-x', 0))
    if slide_mpp <= 0:
        skipped.append((sample_id, 'MPP not found in image header'))
        slide.close()
        continue

    downsample_20x = TARGET_MPP / slide_mpp
    downsample_5x = downsample_20x * CONTEXT_SCALE

    full_patch_20x = int(PATCH_SIZE * downsample_20x)
    full_patch_5x = int(PATCH_SIZE * downsample_5x)

    target_w = int(slide_w / downsample_20x)
    target_h = int(slide_h / downsample_20x)
    n_patches_x = target_w // PATCH_SIZE
    n_patches_y = target_h // PATCH_SIZE

    if n_patches_x == 0 or n_patches_y == 0:
        skipped.append((sample_id, f'slide too small: {slide_w}x{slide_h}, mpp={slide_mpp:.4f}'))
        slide.close()
        continue

    # spot 좌표 (level0 pixel)
    if 'pxl_col_in_fullres' in adata.obs.columns and 'pxl_row_in_fullres' in adata.obs.columns:
        coords_x = adata.obs['pxl_col_in_fullres'].values.astype(float)
        coords_y = adata.obs['pxl_row_in_fullres'].values.astype(float)
    else:
        coords = adata.obsm['spatial']
        coords_x = coords[:, 0].astype(float)
        coords_y = coords[:, 1].astype(float)

    coords_20x_x = coords_x / downsample_20x
    coords_20x_y = coords_y / downsample_20x

    # 존재하는 gene만 추출 후 full matrix에 매핑
    if hasattr(adata.X, 'toarray'):
        avail_expr = adata.X[:, available_gene_indices].toarray().astype(np.float32)
    else:
        avail_expr = np.array(adata.X[:, available_gene_indices], dtype=np.float32)

    expr_matrix = np.zeros((adata.n_obs, NUM_GENES), dtype=np.float32)
    expr_matrix[:, available_marker_indices] = avail_expr

    if hasattr(adata.X, 'toarray'):
        total_counts = np.array(adata.X.sum(axis=1)).flatten().astype(np.float32)
    else:
        total_counts = adata.X.sum(axis=1).astype(np.float32)
    cp10k = expr_matrix / (total_counts[:, None] + 1e-6) * 1e4

    def save_patch(px, py):
        patch_x0 = px * PATCH_SIZE
        patch_y0 = py * PATCH_SIZE

        in_patch = (
            (coords_20x_x >= patch_x0) & (coords_20x_x < patch_x0 + PATCH_SIZE) &
            (coords_20x_y >= patch_y0) & (coords_20x_y < patch_y0 + PATCH_SIZE)
        )
        n_in = in_patch.sum()
        if n_in < MIN_SPOTS:
            return False

        patch_expr = expr_matrix[in_patch]
        patch_cp10k = cp10k[in_patch]

        # Head A: Proportion
        label_prop = (patch_expr > 0).sum(axis=0).astype(np.float32) / n_in

        # Head B: Intensity
        label_int = np.zeros(NUM_GENES, dtype=np.float32)
        for gi in range(NUM_GENES):
            pos_mask = patch_expr[:, gi] > 0
            if pos_mask.sum() > 0:
                label_int[gi] = np.log1p(patch_cp10k[pos_mask, gi]).mean()
        label_int = np.clip(label_int, 0, 10.0) / 10.0

        # 20x patch
        full_x = int(px * full_patch_20x)
        full_y = int(py * full_patch_20x)
        img_region = slide.read_region((full_x, full_y), 0, (full_patch_20x, full_patch_20x))
        img_patch = img_region.convert('RGB').resize((PATCH_SIZE, PATCH_SIZE), Image.LANCZOS)

        # 5x context patch
        center_x = full_x + full_patch_20x // 2
        center_y = full_y + full_patch_20x // 2
        half_5x = full_patch_5x // 2
        x5 = int(np.clip(center_x - half_5x, 0, max(0, slide_w - full_patch_5x)))
        y5 = int(np.clip(center_y - half_5x, 0, max(0, slide_h - full_patch_5x)))
        img_region_5x = slide.read_region((x5, y5), 0, (full_patch_5x, full_patch_5x))
        img_patch_5x = img_region_5x.convert('RGB').resize((PATCH_SIZE, PATCH_SIZE), Image.LANCZOS)

        patch_name = f'{sample_id}_{px}_{py}'
        img_patch.save(f'{save_image_dir}/{st_tech}/{organ_name}/{patch_name}.tiff')
        img_patch_5x.save(f'{save_image_5x_dir}/{st_tech}/{organ_name}/{patch_name}.tiff')
        np.save(f'{save_proportion_dir}/{st_tech}/{organ_name}/{patch_name}.npy', label_prop)
        np.save(f'{save_intensity_dir}/{st_tech}/{organ_name}/{patch_name}.npy', label_int)
        return True

    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = []
        for py in range(n_patches_y):
            for px in range(n_patches_x):
                futures.append(executor.submit(save_patch, px, py))
        for f in futures:
            if f.result():
                patch_count += 1

    slide.close()
    processed.append((sample_id, patch_count))
    print(f'{sample_id} | {st_tech}/{organ_name} | MPP={slide_mpp:.4f} | ds={downsample_20x:.2f} | {patch_count} patches')

print(f'\n=== Done ===')
print(f'Processed: {len(processed)} slides')
print(f'Skipped: {len(skipped)} slides')
for s_id, reason in skipped:
    print(f'  SKIP {s_id}: {reason}')

In [ ]:
# ===== 5. 처리 결과 요약 =====
if processed:
    total_patches = sum(cnt for _, cnt in processed)
    print(f'Total processed slides: {len(processed)}')
    print(f'Total patches saved: {total_patches}')
    print(f'Average patches per slide: {total_patches / len(processed):.1f}')
    print(f'\nTop 10 slides by patch count:')
    for s_id, cnt in sorted(processed, key=lambda x: x[1], reverse=True)[:10]:
        print(f'  {s_id}: {cnt} patches')

for tech in TARGET_TECHS:
    n_files = len(glob(f'{save_base}/image/{tech}/**/*.tiff', recursive=True))
    print(f'\n{tech}: {n_files} image patches saved')

In [ ]:
# ===== 6. 시각화: 샘플 슬라이드 확인 =====
from matplotlib.colors import Normalize
from matplotlib import cm

def visualize_slide_mpp(meta_row, data_path, marker_genes,
                        target_mpp=0.5, patch_size=512, thumb_long_side=2000):
    sample_id = meta_row['id']
    image_file = meta_row['image_filename']

    adata = ad.read_h5ad(f'{data_path}st/{sample_id}.h5ad')
    adata.var_names_make_unique()
    gene_names = adata.var_names.tolist()

    slide = openslide.open_slide(f'{data_path}wsis/{image_file}')
    slide_w, slide_h = slide.dimensions

    slide_mpp = float(slide.properties.get('openslide.mpp-x', 0))
    if slide_mpp <= 0:
        native_mag = float(str(meta_row['magnification']).replace('x', '').replace('X', ''))
        slide_mpp = 10.0 / native_mag
    downsample_20x = target_mpp / slide_mpp

    aspect = slide_w / slide_h
    if aspect >= 1:
        tw, th = thumb_long_side, int(thumb_long_side / aspect)
    else:
        tw, th = int(thumb_long_side * aspect), thumb_long_side
    thumbnail = slide.get_thumbnail((tw, th))
    tw, th = thumbnail.size
    scale_x = tw / slide_w
    scale_y = th / slide_h

    if 'pxl_col_in_fullres' in adata.obs.columns and 'pxl_row_in_fullres' in adata.obs.columns:
        coords_x = adata.obs['pxl_col_in_fullres'].values.astype(float)
        coords_y = adata.obs['pxl_row_in_fullres'].values.astype(float)
    else:
        coords = adata.obsm['spatial']
        coords_x = coords[:, 0].astype(float)
        coords_y = coords[:, 1].astype(float)

    gene_indices = [gene_names.index(g) for g in marker_genes]
    if hasattr(adata.X, 'toarray'):
        expr_matrix = adata.X[:, gene_indices].toarray().astype(np.float32)
        total_counts = np.array(adata.X.sum(axis=1)).flatten().astype(np.float32)
    else:
        expr_matrix = np.array(adata.X[:, gene_indices], dtype=np.float32)
        total_counts = adata.X.sum(axis=1).astype(np.float32)
    cp10k = expr_matrix / (total_counts[:, None] + 1e-6) * 1e4

    coords_20x_x = coords_x / downsample_20x
    coords_20x_y = coords_y / downsample_20x
    full_patch_20x = int(patch_size * downsample_20x)
    n_patches_x = int(slide_w / downsample_20x) // patch_size
    n_patches_y = int(slide_h / downsample_20x) // patch_size

    patch_info = {}
    for py in range(n_patches_y):
        for px in range(n_patches_x):
            x0, y0 = px * patch_size, py * patch_size
            in_patch = (
                (coords_20x_x >= x0) & (coords_20x_x < x0 + patch_size) &
                (coords_20x_y >= y0) & (coords_20x_y < y0 + patch_size)
            )
            cnt = int(in_patch.sum())
            if cnt >= 1:
                patch_info[(px, py)] = (cnt, in_patch)

    genes_to_show = ['EPCAM', 'CD68', 'COL1A1']
    cmaps_show = ['Reds', 'Blues', 'Greens']

    fig, axes = plt.subplots(3, 3, figsize=(28, 26))

    ax = axes[0, 0]
    ax.imshow(thumbnail)
    for px_line in range(n_patches_x + 1):
        ax.axvline(x=px_line * full_patch_20x * scale_x, color='cyan', linewidth=0.4, alpha=0.4)
    for py_line in range(n_patches_y + 1):
        ax.axhline(y=py_line * full_patch_20x * scale_y, color='cyan', linewidth=0.4, alpha=0.4)
    spot_size = max(2, min(30, int(thumb_long_side / 120)))
    if patch_info:
        max_cnt = max(cnt for cnt, _ in patch_info.values())
        norm_cnt = Normalize(vmin=1, vmax=max(max_cnt, 2))
        cmap_patch = cm.YlGn
        for (px, py), (cnt, _) in patch_info.items():
            rx = px * full_patch_20x * scale_x
            ry = py * full_patch_20x * scale_y
            rw = full_patch_20x * scale_x
            rh = full_patch_20x * scale_y
            rect = plt.Rectangle((rx, ry), rw, rh, fill=True,
                                 facecolor=cmap_patch(norm_cnt(cnt)), alpha=0.3,
                                 edgecolor='lime', linewidth=0.8)
            ax.add_patch(rect)
    ax.scatter(coords_x * scale_x, coords_y * scale_y,
               s=spot_size, c='red', alpha=0.6, edgecolors='darkred', linewidths=0.3)
    ax.set_title(f'Grid + Spots | {n_patches_x}x{n_patches_y} grid | '
                 f'{len(patch_info)} valid | {len(coords_x)} spots | MPP={slide_mpp:.4f}',
                 fontsize=12, fontweight='bold')
    ax.axis('off')

    ax = axes[0, 1]
    ax.imshow(thumbnail, alpha=0.4)
    if patch_info:
        for (px, py), (cnt, _) in patch_info.items():
            rx = px * full_patch_20x * scale_x
            ry = py * full_patch_20x * scale_y
            rw = full_patch_20x * scale_x
            rh = full_patch_20x * scale_y
            rect = plt.Rectangle((rx, ry), rw, rh, fill=True,
                                 facecolor=cmap_patch(norm_cnt(cnt)), alpha=0.7,
                                 edgecolor='gray', linewidth=0.3)
            ax.add_patch(rect)
        sm = cm.ScalarMappable(cmap=cmap_patch, norm=norm_cnt)
        sm.set_array([])
        plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04, label='spots per patch')
    ax.set_title('Spots per Patch', fontsize=12, fontweight='bold')
    ax.axis('off')

    ax = axes[0, 2]
    ax.axis('off')
    if patch_info:
        spot_counts = [cnt for cnt, _ in patch_info.values()]
        stats_text = (
            f'Total spots: {len(coords_x)}\n'
            f'Valid patches: {len(patch_info)} / {n_patches_x * n_patches_y}\n'
            f'Spots/patch: min={min(spot_counts)}, med={int(np.median(spot_counts))}, '
            f'max={max(spot_counts)}, mean={np.mean(spot_counts):.1f}\n\n'
            f'Technology: {meta_row["st_technology"]}\n'
            f'Organ: {meta_row["organ"]}\n'
            f'MPP: {slide_mpp:.4f} um/pixel\n'
            f'Downsample: {downsample_20x:.2f}x'
        )
        ax.text(0.1, 0.5, stats_text, transform=ax.transAxes,
                fontsize=14, verticalalignment='center', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        ax.set_title('Summary', fontsize=12, fontweight='bold')

    norm_prop = Normalize(vmin=0, vmax=1)
    for col, (gene, cmap_name) in enumerate(zip(genes_to_show, cmaps_show)):
        ax = axes[1, col]
        ax.imshow(thumbnail, alpha=0.4)
        gi = marker_genes.index(gene)
        for (px, py), (cnt, in_patch) in patch_info.items():
            prop = (expr_matrix[in_patch, gi] > 0).sum() / cnt
            rx = px * full_patch_20x * scale_x
            ry = py * full_patch_20x * scale_y
            rw = full_patch_20x * scale_x
            rh = full_patch_20x * scale_y
            color = cm.get_cmap(cmap_name)(norm_prop(prop))
            rect = plt.Rectangle((rx, ry), rw, rh, fill=True,
                                 facecolor=color, alpha=0.7,
                                 edgecolor='gray', linewidth=0.3)
            ax.add_patch(rect)
        sm = cm.ScalarMappable(cmap=cmap_name, norm=norm_prop)
        sm.set_array([])
        plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04, label='proportion')
        ax.set_title(f'{gene} -- Proportion', fontsize=12, fontweight='bold')
        ax.axis('off')

    norm_int = Normalize(vmin=0, vmax=1)
    for col, (gene, cmap_name) in enumerate(zip(genes_to_show, cmaps_show)):
        ax = axes[2, col]
        ax.imshow(thumbnail, alpha=0.4)
        gi = marker_genes.index(gene)
        for (px, py), (cnt, in_patch) in patch_info.items():
            pos_mask = expr_matrix[in_patch, gi] > 0
            if pos_mask.sum() > 0:
                intensity = np.clip(np.log1p(cp10k[in_patch][pos_mask, gi]).mean() / 10.0, 0, 1)
            else:
                intensity = 0.0
            rx = px * full_patch_20x * scale_x
            ry = py * full_patch_20x * scale_y
            rw = full_patch_20x * scale_x
            rh = full_patch_20x * scale_y
            color = cm.get_cmap(cmap_name)(norm_int(intensity))
            rect = plt.Rectangle((rx, ry), rw, rh, fill=True,
                                 facecolor=color, alpha=0.7,
                                 edgecolor='gray', linewidth=0.3)
            ax.add_patch(rect)
        sm = cm.ScalarMappable(cmap=cmap_name, norm=norm_int)
        sm.set_array([])
        plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04, label='intensity (norm)')
        ax.set_title(f'{gene} -- Intensity', fontsize=12, fontweight='bold')
        ax.axis('off')

    plt.suptitle(f'{sample_id}  |  {meta_row["st_technology"]} / {meta_row["organ"]}',
                 fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()
    slide.close()


visualize_slide_mpp(target_df.iloc[0], data_path, MARKER_GENES,
                    target_mpp=TARGET_MPP, patch_size=PATCH_SIZE)